# Model evaluation with `polarbearings` — scalar metrics and plot-ready curves

`polarbearings` gives you model-evaluation metrics as native Polars expressions and
the common diagnostic curves as tidy, plot-ready **LazyFrames**. Several things fall
straight out of that design, shown in order below:

1. **Scalars** — compute many metrics in one pass, collect them alongside a curve
   in a single batch, and segment any of it with `group_by`.
2. **Confidence intervals** — `bootstrap_ci` wraps any metric (Bayesian bootstrap,
   in-engine), whole-frame or per group.
3. **Sample weights** — every metric and curve takes an optional `weight=`.
4. **Curves** — `roc_curve`, `pr_curve`, `det_curve`, `expected_cost`, and
   `calibration_curve`, each one call returning data you hand straight to Plotly.

Because these are all just expressions and weighted curves, they **compose** — the
notebook ends by stacking weighted ROC curves under bootstrap resamples into a
confidence band. All four threshold curves are thin column-math wrappers over one
primitive, `confusion_curve`, which returns `(threshold, tp, fp, fn, tn)` at every
distinct score in one sorted pass — or, with `thresholds=`, at a fixed grid of
operating points. Everything stays lazy and single-pass.

> Run with the `notebooks` dependency group:
> `uv run --group notebooks jupyter lab notebooks/diagnostics.ipynb`

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import polarbearings as pb
from polarbearings.calibration import calibration_curve

pio.renderers.default = "notebook_connected"  # CDN plotly.js -> lighter file

rng = np.random.default_rng(0)
n = 4_000
cohort = rng.choice(["A", "B", "C"], n)

# Per cohort: (base rate, positive-score loc, negative-score loc). A is clean and
# well separated; B is moderate. C is the "risky" cohort — its scores sit HIGH but
# its signal is reversed (positives score *lower* than negatives), so the model has
# C roughly backwards and its mass lives in the high-score region. C is a minority
# of rows but business-critical, so we upweight it 4x — which (further down) bends
# the pooled ROC into a non-concave hook.
spec = {"A": (0.50, 2.0, -1.2), "B": (0.45, 1.0, -0.6), "C": (0.38, 1.0, 1.8)}
p = np.array([spec[c][0] for c in cohort])
pos_loc = np.array([spec[c][1] for c in cohort])
neg_loc = np.array([spec[c][2] for c in cohort])
y = (rng.random(n) < p).astype(int)
loc = np.where(y == 1, pos_loc, neg_loc)
score = 1 / (1 + np.exp(-rng.normal(loc, 0.8)))  # sigmoid logits, in [0, 1]
weight = np.where(cohort == "C", 4.0, 1.0)
df = pl.DataFrame({"cohort": cohort, "y": y, "score": score, "weight": weight})
df.head()

cohort,y,score,weight
str,i64,f64,f64
"""C""",0,0.781549,4.0
"""B""",1,0.872235,1.0
"""B""",0,0.214375,1.0
"""A""",1,0.839712,1.0
"""A""",1,0.919883,1.0


## Scalars: many at once, with curves, and per group

Every scalar metric is a Polars expression, so you compute as many as you like in
one `select` — a single pass over the data. A curve is a different shape (one row
per threshold), but it's lazy too, so `pl.collect_all` runs the scalars and the
curve together as one batch. And anything here drops into `group_by` for per-segment
results.

In [2]:
# Many metrics, one pass over the data — each is just a Polars expression.
df.select(
    pb.roc_auc("y", "score").alias("auc"),
    pb.average_precision("y", "score").alias("ap"),
    pb.log_loss("y", "score").alias("log_loss"),
    pb.brier_score("y", "score").alias("brier"),
    pb.precision("y", "score", threshold=0.5).alias("precision@0.5"),
    pb.recall("y", "score", threshold=0.5).alias("recall@0.5"),
    pb.f1_score("y", "score", threshold=0.5).alias("f1@0.5"),
)

auc,ap,log_loss,brier,precision@0.5,recall@0.5,f1@0.5
f64,f64,f64,f64,f64,f64,f64
0.742761,0.600091,0.692872,0.227489,0.612435,0.93043,0.738662


In [3]:
# A scalar summary (one row) and a curve (many rows) are different shapes, but both
# are lazy — collect them together so Polars plans them as a single batch.
scalars = df.lazy().select(
    pb.roc_auc("y", "score").alias("auc"),
    pb.average_precision("y", "score").alias("ap"),
)
roc = pb.roc_curve(df, "y", "score")

summary, roc_df = pl.collect_all([scalars, roc])
print(summary)
roc_df.head()

shape: (1, 2)
┌──────────┬──────────┐
│ auc      ┆ ap       │
│ ---      ┆ ---      │
│ f64      ┆ f64      │
╞══════════╪══════════╡
│ 0.742761 ┆ 0.600091 │
└──────────┴──────────┘


threshold,fpr,tpr
f64,f64,f64
inf,0.0,0.0
0.987955,0.0,0.000566
0.985994,0.000448,0.000566
0.985031,0.000896,0.000566
0.981202,0.000896,0.001131


**Per group.** Drop the same expressions into `group_by().agg(...)` for one row
per segment in a single pass; the curve helpers take `by=` for the same effect.

In [4]:
# The same expressions inside group_by give one row per segment, one pass.
df.group_by("cohort").agg(
    pb.roc_auc("y", "score").alias("auc"),
    pb.average_precision("y", "score").alias("ap"),
    pb.f1_score("y", "score", threshold=0.5).alias("f1@0.5"),
    pl.len().alias("n"),
).sort("cohort")

cohort,auc,ap,f1@0.5,n
str,f64,f64,f64,u32
"""A""",0.997506,0.997636,0.965986,1317
"""B""",0.919351,0.908317,0.820166,1331
"""C""",0.249624,0.259667,0.511654,1352


In [5]:
# A separate ROC per cohort, all in one pass — every curve helper takes `by=`.
pb.roc_curve(df, "y", "score", by="cohort").collect().head()

cohort,threshold,fpr,tpr
str,f64,f64,f64
"""A""",inf,0.0,0.0
"""A""",0.987955,0.0,0.001548
"""A""",0.981202,0.0,0.003096
"""A""",0.98074,0.0,0.004644
"""A""",0.980133,0.0,0.006192


## Confidence intervals (bootstrap)

Because every metric accepts a `weight`, a bootstrap replicate is just the metric
under a random weight vector — `polarbearings` uses the **Bayesian bootstrap**,
generated in-engine, so there's no Python resampling loop. `bootstrap_ci` returns
`{estimate, low, high}` for the whole frame; pass `by=` for one interval per group
(a single vectorized pass, not a loop). It works for *any* metric — swap `roc_auc`
for `average_precision`, `f1_score`, `rmse`, ….

In [6]:
from polarbearings import bootstrap_ci

# Whole-frame CI: {estimate, low, high}. method="bc" = bias-corrected.
ci = bootstrap_ci(df, pb.roc_auc, "y", "score", n_resamples=1000, method="bc")
print(f"ROC AUC = {ci['estimate']:.3f}  (95% CI: {ci['low']:.3f}–{ci['high']:.3f})")

ROC AUC = 0.743  (95% CI: 0.728–0.758)


In [7]:
# One interval per group — the cohorts separate cleanly (A best, C worst).
auc_by = bootstrap_ci(df, pb.roc_auc, "y", "score", by="cohort", n_resamples=1000)
auc_by

cohort,estimate,low,high
str,f64,f64,f64
"""A""",0.997506,0.995705,0.998739
"""B""",0.919351,0.905484,0.933994
"""C""",0.249624,0.222294,0.277875


In [8]:
# Per-cohort estimates with their CIs as asymmetric error bars — bootstrap_ci's
# Polars frame feeds plotly.express directly; the error bars are just columns.
auc_err = auc_by.with_columns(
    err_hi=pl.col("high") - pl.col("estimate"),
    err_lo=pl.col("estimate") - pl.col("low"),
)
fig = px.scatter(
    auc_err, x="cohort", y="estimate", error_y="err_hi", error_y_minus="err_lo",
    title="ROC AUC by cohort (95% bootstrap CI)", labels={"estimate": "ROC AUC"},
)
fig.update_traces(marker_size=11)
fig.update_layout(width=520, height=420)
fig

## Sample weights

Every metric and curve takes an optional `weight=` (a column name or expression);
leave it off for the unweighted result. Here `weight` upweights the risky cohort
(C) 4×. Because the model has C roughly backwards — high scores, mostly negatives —
weighting doesn't just lower the pooled metrics, it **changes the shape of the
curve**: the weighted ROC develops a non-concave *hook*, dipping below the diagonal
at high thresholds (where C's high-scored negatives pile up as false positives)
before recovering. The unweighted curve is an ordinary concave ROC.

In [9]:
# Same metrics, with and without weights — `weight=` is the only difference.
pl.concat([
    df.select(
        pl.lit("unweighted").alias("variant"),
        pb.roc_auc("y", "score").alias("auc"),
        pb.average_precision("y", "score").alias("ap"),
        pb.brier_score("y", "score").alias("brier"),
    ),
    df.select(
        pl.lit("weighted").alias("variant"),
        pb.roc_auc("y", "score", weight="weight").alias("auc"),
        pb.average_precision("y", "score", weight="weight").alias("ap"),
        pb.brier_score("y", "score", weight="weight").alias("brier"),
    ),
])

variant,auc,ap,brier
str,f64,f64,f64
"""unweighted""",0.742761,0.600091,0.227489
"""weighted""",0.504542,0.385934,0.353299


In [10]:
from polarbearings import roc_curve

# The curve honors weights too. Tag each frame and concat — plotly.express then
# colors by the label in a single call, no per-trace code.
roc_unw = roc_curve(df, "y", "score").collect().with_columns(weighting=pl.lit("unweighted"))
roc_w = (
    roc_curve(df, "y", "score", weight="weight")
    .collect()
    .with_columns(weighting=pl.lit("weighted (C×4)"))
)
fig = px.line(
    pl.concat([roc_unw, roc_w]), x="fpr", y="tpr", color="weighting",
    title="Weighted vs unweighted ROC",
    labels={"fpr": "False positive rate", "tpr": "True positive rate"},
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=560, height=480)
fig

**Importance vs frequency weights (when bootstrapping).** A `weight` can mean two
things. As an *importance* weight, a bootstrap replicate scales it (`weight · draw`).
But if it's a *frequency / dedup count* — the row stands in for that many identical
cases — the count should **parameterize** the draw (`Gamma(w, 1)`), not scale it:
scaling over-disperses (variance `w²` vs `w`), so the CI comes out too wide. Pass
`weight_kind="frequency"` for the correct, tighter interval:

In [11]:
# Bootstrap the AUC CI treating `weight` as importance vs. as a dedup case count.
boot = df.with_row_index("id")

def auc_ci(weight_kind, n=400):
    reps = pl.collect_all([
        boot.lazy().select(pb.roc_auc(
            "y", "score",
            weight=pb.bootstrap_weight("id", seed=b, weight="weight", weight_kind=weight_kind),
        ).alias("a"))
        for b in range(n)
    ])
    a = np.array([r["a"][0] for r in reps])
    return a.mean(), np.quantile(a, 0.025), np.quantile(a, 0.975)

for wk in ["importance", "frequency"]:
    m, lo, hi = auc_ci(wk)
    print(f"{wk:11s}  AUC {m:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   width {hi - lo:.3f}")

importance   AUC 0.505   95% CI [0.487, 0.525]   width 0.038


frequency    AUC 0.504   95% CI [0.493, 0.517]   width 0.025


## The primitive: `confusion_curve`

Every threshold-based curve (ROC, PR, DET, cost) is a function of the four
confusion cells `tp, fp, fn, tn`. `confusion_curve` computes them at **every
distinct score** in a single sorted `O(n log n)` pass — the exact step function —
returning one tidy row per threshold. Pass `thresholds=` to instead evaluate a
**fixed grid** of operating points (comparable across models, great for
monitoring), computed in one pass and still fully lazy. The curve helpers below are
column math over this frame, so you rarely call it directly — but here it is.

In [12]:
from polarbearings import confusion_curve

# Exact: one tidy row per distinct score, single sorted pass.
exact = confusion_curve(df, "y", "score").collect()

# Grid: thresholds=N picks N data-driven operating points (score quantiles),
# still one pass. Pass a list[float] or a spec instead for full control.
grid = confusion_curve(df, "y", "score", thresholds=20).collect()

print(f"exact: {exact.height} rows   grid: {grid.height} rows")
exact.head()

exact: 4001 rows   grid: 20 rows


threshold,tp,fp,fn,tn
f64,i64,i64,i64,i64
inf,0,0,1768,2232
0.987955,1,0,1767,2232
0.985994,1,1,1767,2231
0.985031,1,2,1767,2230
0.981202,2,2,1766,2230


## ROC / AUC

`roc_curve` returns `(threshold, fpr, tpr)` in plot order — no manual
`tp / (tp + fn)` math. The scalar summary is `pb.roc_auc`.

In [13]:
from polarbearings import roc_curve

auc = df.select(pb.roc_auc("y", "score")).item()
roc = roc_curve(df, "y", "score").collect()

# The curve frame goes straight into plotly.express — column names do the rest.
fig = px.line(
    roc, x="fpr", y="tpr", hover_data=["threshold"],
    title=f"ROC curve (AUC = {auc:.3f})",
    labels={"fpr": "False positive rate", "tpr": "True positive rate"},
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=560, height=480)
fig

Under the hood `roc_curve` calls `confusion_curve` and forms the two rates. Pass
`thresholds=` to it as well for a fixed grid instead of the exact curve (see the
last section), and `by=` for a separate ROC per segment in one pass.

## Precision–Recall

`pr_curve` returns `(threshold, precision, recall)`. The scalar summary is
`pb.average_precision`.

In [14]:
from polarbearings import pr_curve

ap = df.select(pb.average_precision("y", "score")).item()
pr = pr_curve(df, "y", "score").collect().sort("recall")

fig = px.line(
    pr, x="recall", y="precision", hover_data=["threshold"],
    title=f"Precision-Recall (AP = {ap:.3f})",
    labels={"recall": "Recall", "precision": "Precision"},
)
base = float((df["y"] == 1).mean())
fig.add_hline(y=base, line_dash="dash", line_color="gray",
              annotation_text=f"baseline = {base:.2f}")
fig.update_layout(width=560, height=480)
fig

## Detection Error Tradeoff (DET)

`det_curve` returns `(threshold, fpr, fnr)` — false-positive vs false-negative
rate. Mirrors `sklearn.metrics.det_curve` (the trivial endpoints are omitted), and
is usually viewed on probit-scaled axes.

In [15]:
from polarbearings import det_curve

det = det_curve(df, "y", "score").collect()

fig = px.line(
    det, x="fpr", y="fnr", hover_data=["threshold"],
    title="Detection Error Tradeoff",
    labels={"fpr": "False positive rate", "fnr": "False negative rate"},
)
fig.update_layout(width=560, height=480)
fig

## Cost curves

`expected_cost` applies a per-cell cost (or benefit) to the confusion cells and
sweeps the total across thresholds — the cost-optimal operating point is the
`cost` argmin. Here a false negative costs 5, a false positive costs 1, and correct
calls are free. A fixed `thresholds=` grid keeps the x-axis clean.

In [16]:
from polarbearings import expected_cost

costs = {"fp": 1.0, "fn": 5.0}
cost = expected_cost(
    df, "y", "score", costs, thresholds=list(np.linspace(0, 1, 101))
).collect()
best = cost.sort("cost").row(0, named=True)

fig = px.line(
    cost, x="threshold", y="cost", title="Total cost vs threshold (FN=5, FP=1)",
    labels={"threshold": "threshold", "cost": "total cost"},
)
fig.add_vline(x=best["threshold"], line_dash="dash", line_color="crimson",
              annotation_text=f"min @ {best['threshold']:.2f}")
fig.update_layout(width=620, height=420)
fig

In [17]:
# The fp/fn behind the cost come from the same grid of cells.
cells = confusion_curve(df, "y", "score", thresholds=list(np.linspace(0, 1, 101))).collect()
print(f"min-cost threshold = {best['threshold']:.2f}  (total cost = {best['cost']:,.0f})")
cost.join(cells.select("threshold", "fp", "fn"), on="threshold").filter(
    pl.col("threshold").is_between(0.30, 0.45)
)

min-cost threshold = 0.42  (total cost = 1,449)


threshold,cost,fp,fn
f64,f64,i64,i64
0.45,1523.0,1133,78
0.44,1516.0,1151,73
0.43,1472.0,1172,60
0.42,1449.0,1194,51
0.41,1467.0,1222,49
…,…,…,…
0.34,1532.0,1407,25
0.33,1543.0,1443,20
0.32,1565.0,1470,19


`expected_cost` also takes `normalize=True` for expected cost *per decision*, and
`by=` for a separate cost curve per segment — all in one pass.

## Calibration (already a helper)

`calibration_curve` ships today and returns plot-ready tidy data directly, with a
choice of binning strategy. Below: equal-width vs equal-frequency bins, overlaid on
the reliability diagonal.

In [18]:
uniform = calibration_curve(df, "y", "score", n_bins=10, strategy="uniform").collect()
quantile = calibration_curve(df, "y", "score", n_bins=10, strategy="quantile").collect()
uniform

bin,bin_lower,bin_upper,count,prob_pred,prob_true
i64,f64,f64,u32,f64,f64
0,0.0,0.1,96,0.069273,0.0
1,0.1,0.2,273,0.156431,0.007326
2,0.2,0.3,338,0.248129,0.029586
3,0.3,0.4,318,0.347063,0.100629
4,0.4,0.5,289,0.448706,0.273356
5,0.5,0.6,285,0.55187,0.519298
6,0.6,0.7,360,0.655919,0.666667
7,0.7,0.8,551,0.755809,0.635209
8,0.8,0.9,878,0.852266,0.627563


In [19]:
# Tag each binning strategy and concat — plotly.express draws both in one call.
cal = pl.concat([
    uniform.with_columns(binning=pl.lit("uniform bins")),
    quantile.with_columns(binning=pl.lit("quantile bins")),
])
fig = px.line(
    cal, x="prob_pred", y="prob_true", color="binning", markers=True,
    title="Calibration (reliability) curve",
    labels={"prob_pred": "Mean predicted probability",
            "prob_true": "Observed positive fraction"},
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=560, height=480)
fig

In [20]:
# Custom edges (no scikit-learn equivalent): fixed, comparable score bands.
calibration_curve(df, "y", "score", bins=[0.0, 0.25, 0.5, 0.75, 1.0]).collect()

bin,bin_lower,bin_upper,count,prob_pred,prob_true
i64,f64,f64,u32,f64,f64
0,0.0,0.25,544,0.162737,0.011029
1,0.25,0.5,770,0.369787,0.151948
2,0.5,0.75,869,0.639777,0.609896
3,0.75,1.0,1817,0.86583,0.613649


**By cohort.** Like the other curves, `calibration_curve` takes `by=` — one
reliability curve per group in a single pass, with shared (global) bins so the
groups line up. Cohort C sits far below the diagonal in the high-score region: its
confident predictions don't pan out (the model has C reversed) — which is exactly
why upweighting C bent the ROC into a hook earlier.

In [21]:
# Calibration by cohort — one reliability curve per group, shared bins, one pass.
# calibration_curve's `by=` output drops straight into plotly.express via color=.
cal_by = calibration_curve(df, "y", "score", n_bins=12, by="cohort").collect()

fig = px.line(
    cal_by.sort("prob_pred"), x="prob_pred", y="prob_true", color="cohort", markers=True,
    title="Calibration by cohort",
    labels={"prob_pred": "Mean predicted probability",
            "prob_true": "Observed positive fraction"},
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=560, height=480)
fig

In [22]:
# Quantify what the curves show: ECE/MCE are scalar expressions, so they go right
# into group_by. Cohort C's high error matches its far-from-diagonal curve.
df.group_by("cohort").agg(
    ece=pb.expected_calibration_error("y", "score", n_bins=10),
    mce=pb.maximum_calibration_error("y", "score", n_bins=10),
    n=pl.len(),
).sort("cohort")

cohort,ece,mce,n
str,f64,f64,u32
"""A""",0.179773,0.378278,1317
"""B""",0.14563,0.251313,1331
"""C""",0.477769,0.841128,1352


## Exact vs grid: the `thresholds=` argument

Every curve helper forwards `thresholds=` to `confusion_curve`, controlling the
operating points:

- `None` (default) — **exact** curve, one vertex per distinct score (one sorted pass)
- an **int** `N` — `N` data-driven thresholds at score **quantiles**; adapts to the
  distribution, and under `by=` each group uses its own
- a **spec** — `quantiles(N)`, `equal_width(N)`, or `linspace(N, lo, hi)` for explicit control
- a **list[float]** — exactly those fixed thresholds

Below, the exact ROC overlaid with `thresholds=20` (20 quantile points — note how
they cluster where scores are dense).

In [23]:
exact_roc = roc_curve(df, "y", "score").collect()
grid_roc = roc_curve(df, "y", "score", thresholds=20).collect()  # 20 quantile points

# Express for the exact line; one go marker trace overlaid for the grid points
# (both consume the Polars columns directly — no .to_list()).
fig = px.line(
    exact_roc, x="fpr", y="tpr", title="Exact vs grid ROC (thresholds= argument)",
    labels={"fpr": "False positive rate", "tpr": "True positive rate"},
)
fig.data[0].update(name=f"exact ({exact_roc.height} pts)", showlegend=True)
fig.add_scatter(x=grid_roc["fpr"], y=grid_roc["tpr"], mode="markers",
                name="grid (thresholds=20)", marker_size=7)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=560, height=480)
fig

## Composing it all: a bootstrapped ROC band

There's no single `bootstrap_curve` call — `bootstrap()` builds the sampling
distribution of a *scalar* metric. But the pieces compose: `bootstrap_weight(id, seed=b)`
is an in-engine replicate weight, and every curve takes `weight=`. Below we bootstrap
the **weighted** ROC — folding each replicate's `Exp(1)` draw into the cohort weights —
on a **fixed** `thresholds=` grid so replicates share x-points, then summarize per
threshold. Hashing the row `id` makes it reproducible across runs (and works under
`by=`); `pl.collect_all` runs all replicates as one batch. The result shows the
non-concave hook *with* its confidence band.

In [24]:
# Each replicate: the *weighted* ROC under one `bootstrap_weight` replicate (an
# in-engine Exp(1) draw folded into the cohort weights), on a shared threshold grid
# so replicates align. No numpy; hashing the row id makes each replicate
# reproducible across runs. collect_all runs them as one batch.
B, grid = 200, list(np.linspace(0.02, 0.98, 40))
boot = df.with_row_index("id")
replicates = [
    roc_curve(boot, "y", "score", thresholds=grid,
              weight=pb.bootstrap_weight("id", seed=b, weight="weight"))
    for b in range(B)
]
band = (
    pl.concat(pl.collect_all(replicates))
    .group_by("threshold")
    .agg(
        fpr=pl.col("fpr").mean(),
        tpr=pl.col("tpr").mean(),
        tpr_lo=pl.col("tpr").quantile(0.025),
        tpr_hi=pl.col("tpr").quantile(0.975),
    )
    .sort("fpr")
)

fig = go.Figure()
fig.add_trace(go.Scatter(  # 95% band: fill between lo and hi
    x=band["fpr"].to_list() + band["fpr"].to_list()[::-1],
    y=band["tpr_hi"].to_list() + band["tpr_lo"].to_list()[::-1],
    fill="toself", fillcolor="rgba(0,100,200,0.2)", line=dict(color="rgba(0,0,0,0)"),
    name="95% band", hoverinfo="skip",
))
fig.add_trace(go.Scatter(x=band["fpr"].to_list(), y=band["tpr"].to_list(),
                         mode="lines", name="mean weighted ROC", line=dict(color="rgb(0,100,200)")))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(title=f"Bootstrapped weighted ROC ({B} resamples, 95% band) — note the hook",
                  xaxis_title="False positive rate", yaxis_title="True positive rate",
                  width=560, height=480)
fig

## Summary

| Plot | Helper | Scalar summary |
|---|---|---|
| ROC / AUC | `roc_curve` | `roc_auc` |
| Precision–Recall | `pr_curve` | `average_precision` |
| Detection Error Tradeoff | `det_curve` | — |
| Cost curve | `expected_cost` | argmin → optimal threshold |
| Calibration | `calibration_curve` | `expected_calibration_error` / `maximum_calibration_error` |

Each threshold curve is one call returning tidy, plot-ready data. All four are thin
column-math wrappers over `confusion_curve`, which yields `(threshold, tp, fp, fn,
tn)` either **exactly** (every distinct score, one sorted pass) or on a fixed
`thresholds=` **grid**. Everything stays lazy and single-pass, and `by=` computes a
separate curve per segment in the same pass.